# 1. Introduction
## Feature Engineering

### Objectif

Cette étape consiste à créer des variables explicatives pertinentes à partir des données historiques des commandes.

Le but est d’améliorer les performances des modèles prédictifs en capturant :
- les comportements temporels ;
- les habitudes de commande ;
- les tendances des stations-service.

# 2. Importation des Bibliothèques

In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

# Configuration affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)

# 3. Chargement des données propres

In [3]:
df = pd.read_excel("../data/propre/commandes_propres.xlsx")

print(df.shape)

display(df.head())

(34150, 13)


,Date comptabilisation,N° article,N° origine,N° contact origine,Quantité facturée,Code Produit,Type document,NOM CLIENT,Nom sous compte,TYPE,STATUT,Code Station,GROUPE TYPE
0,2020-01-13,A0001,C1234,CC2616,5000,PROD-BLANC,Expédition vente,GIBLAND COMPAGNY,STATION ATTAKE,CODO,Fonctionnelle,ST0049,STATION
1,2020-01-13,A0002,C1234,CC2616,4000,PROD-BLANC,Expédition vente,GIBLAND COMPAGNY,STATION ATTAKE,CODO,Fonctionnelle,ST0049,STATION
2,2020-01-13,A0001,C1249,CC2650,1000,PROD-BLANC,Expédition vente,TONI ET FILS,STATION NDALI,DODO,Fonctionnelle,ST0092,STATION
3,2020-01-13,A0002,C1249,CC2650,16000,PROD-BLANC,Expédition vente,TONI ET FILS,STATION NDALI,DODO,Fonctionnelle,ST0092,STATION
4,2020-01-13,A0001,C1233,CC2614,5000,PROD-BLANC,Expédition vente,AYODELE BUSINESS,STATION OUIDAH,CODO,Fonctionnelle,ST0016,STATION


# 4. Vérification du format des dates
Les modèles temporels ont besoin : de dates exploitables et d’ordre chronologique.

In [4]:
# CONVERSION DES DATES

df["Date comptabilisation"] = pd.to_datetime(
    df["Date comptabilisation"]
)

print(df["Date comptabilisation"].dtype)

datetime64[ns]


In [5]:
# TRI CHRONOLOGIQUE
# Les variables historiques dépendent : de l’ordre temporel et des commandes précédentes.

df = df.sort_values(
    ["Code Station", "Date comptabilisation"]
)

display(df.head())

,Date comptabilisation,N° article,N° origine,N° contact origine,Quantité facturée,Code Produit,Type document,NOM CLIENT,Nom sous compte,TYPE,STATUT,Code Station,GROUPE TYPE
13322,2022-09-27,A0001,C0083,CC6471,20000,PROD-BLANC,Expédition vente,2M ASSOCIES,STATION BOULEVARD MARINA,CODO,Fonctionnelle,ST0001,STATION
13323,2022-09-27,A0002,C0083,CC6471,15000,PROD-BLANC,Expédition vente,2M ASSOCIES,STATION BOULEVARD MARINA,CODO,Fonctionnelle,ST0001,STATION
13637,2022-10-17,A0001,C0083,CC6471,8000,PROD-BLANC,Expédition vente,2M ASSOCIES,STATION BOULEVARD MARINA,CODO,Fonctionnelle,ST0001,STATION
13648,2022-10-17,A0002,C0083,CC6471,10000,PROD-BLANC,Expédition vente,2M ASSOCIES,STATION BOULEVARD MARINA,CODO,Fonctionnelle,ST0001,STATION
13727,2022-10-20,A0002,C0083,CC6471,8000,PROD-BLANC,Expédition vente,2M ASSOCIES,STATION BOULEVARD MARINA,CODO,Fonctionnelle,ST0001,STATION


In [6]:
# AGREGER LES COMMANDES JOURNALIERES

df_daily = (
    df.groupby(
        [
            "Date comptabilisation",
            "Code Station",
            "N° article"
        ]
    )
    .agg({
        "Quantité facturée": "sum"
    })
    .reset_index()
    .copy()
)

all_dates = pd.date_range(
    start=df_daily["Date comptabilisation"].min(),
    end=df_daily["Date comptabilisation"].max(),
    freq="D"
)

df_daily = df_daily.sort_values(
    [
        "Code Station",
        "N° article",
        "Date comptabilisation"
    ]
).copy()

display(df_daily.head(20))

,Date comptabilisation,Code Station,N° article,Quantité facturée
10703,2022-09-27,ST0001,A0001,20000
10959,2022-10-17,ST0001,A0001,8000
11042,2022-10-24,ST0001,A0001,6000
11079,2022-10-25,ST0001,A0001,10000
11143,2022-11-03,ST0001,A0001,15000
11228,2022-11-08,ST0001,A0001,12000
11283,2022-11-11,ST0001,A0001,22000
12009,2023-01-05,ST0001,A0001,10000
12073,2023-01-12,ST0001,A0001,12000
12110,2023-01-16,ST0001,A0001,10000


# 5. Création des variables temporelles


**Agrégation mensuelle**

L'objectif est d'avoir **une ligne par mois, par station et par article**.

Pourquoi mensuel ?
- Les modèles ARIMA/SARIMA travaillent sur des séries mensuelles
- Cela permet une **comparaison équitable** entre modèles ML et séries temporelles
- Réduit le bruit des variations journalières et hebdomadaires

In [7]:
# AGRÉGATION MENSUELLE
# Une ligne = un mois × une station × un article

# Créer la colonne mois (période mensuelle)
df_daily['mois_periode'] = df_daily['Date comptabilisation'].dt.to_period('M')

# Agréger : somme des quantités + nombre de commandes dans le mois
df_monthly = (
    df_daily
    .groupby(['Code Station', 'N° article', 'mois_periode'])
    .agg(
        quantite_mois  = ('Quantité facturée', 'sum'),   # variable cible Y
        nb_commandes   = ('Quantité facturée', 'count'), # nb de jours commandés
    )
    .reset_index()
    .sort_values(['Code Station', 'N° article', 'mois_periode'])
    .reset_index(drop=True)
)

# Convertir la période en timestamp (1er du mois)
df_monthly['mois'] = df_monthly['mois_periode'].dt.to_timestamp()

print(f"Lignes df_daily   : {len(df_daily):,}")
print(f"Lignes df_monthly : {len(df_monthly):,}")
print(f"Mois couverts     : {df_monthly['mois_periode'].nunique()}")
print(f"Stations          : {df_monthly['Code Station'].nunique()}")
print(f"Articles          : {df_monthly['N° article'].nunique()}")
print()
display(df_monthly.head(15))


Lignes df_daily   : 27,071
Lignes df_monthly : 7,631
Mois couverts     : 72
Stations          : 94
Articles          : 2



,Code Station,N° article,mois_periode,quantite_mois,nb_commandes,mois
0,ST0001,A0001,2022-09,20000,1,2022-09-01
1,ST0001,A0001,2022-10,24000,3,2022-10-01
2,ST0001,A0001,2022-11,49000,3,2022-11-01
3,ST0001,A0001,2023-01,56000,5,2023-01-01
4,ST0001,A0001,2023-02,34000,4,2023-02-01
5,ST0001,A0001,2023-03,90000,5,2023-03-01
6,ST0001,A0001,2023-04,48000,2,2023-04-01
7,ST0001,A0001,2023-05,105000,5,2023-05-01
8,ST0001,A0001,2023-06,65000,3,2023-06-01
9,ST0001,A0001,2023-07,95000,4,2023-07-01


# 7. Création des variables temporelles

Ces features donnent au modèle le **contexte calendaire** de chaque mois.

In [8]:
# FEATURES TEMPORELLES — niveau mensuel

# Année — pour capturer la tendance long terme
df_monthly['annee']      = df_monthly['mois'].dt.year

# Mois (1-12) — saisonnalité annuelle
df_monthly['num_mois']   = df_monthly['mois'].dt.month

# Trimestre (1-4)
df_monthly['trimestre']  = df_monthly['mois'].dt.quarter

# Encodage cyclique du mois — le mois 12 et le mois 1 sont proches
# Un encodage linéaire ferait croire que 12 et 1 sont très éloignés
df_monthly['mois_sin']   = np.sin(2 * np.pi * df_monthly['num_mois'] / 12)
df_monthly['mois_cos']   = np.cos(2 * np.pi * df_monthly['num_mois'] / 12)

# Indicateurs binaires métier
df_monthly['est_t1']     = (df_monthly['trimestre'] == 1).astype(int)  # jan-mars
df_monthly['est_t4']     = (df_monthly['trimestre'] == 4).astype(int)  # oct-déc

# Saison climatique Bénin
def saison_benin(mois):
    if mois in [12, 1, 2]:    return 0  # Grande saison sèche
    elif mois in [3, 4]:      return 1  # Petite saison sèche
    elif mois in [5, 6, 7]:   return 2  # Grande saison pluies
    else:                     return 3  # Petite saison pluies

df_monthly['saison_benin'] = df_monthly['num_mois'].apply(saison_benin)

print("Features temporelles créées :")
cols_temp = ['annee','num_mois','trimestre','mois_sin','mois_cos','saison_benin']
display(df_monthly[['mois','Code Station','N° article'] + cols_temp].head(15))


Features temporelles créées :


,mois,Code Station,N° article,annee,num_mois,trimestre,mois_sin,mois_cos,saison_benin
0,2022-09-01,ST0001,A0001,2022,9,3,-1.000000e+00,-1.836970e-16,3
1,2022-10-01,ST0001,A0001,2022,10,4,-8.660254e-01,5.000000e-01,3
2,2022-11-01,ST0001,A0001,2022,11,4,-5.000000e-01,8.660254e-01,3
3,2023-01-01,ST0001,A0001,2023,1,1,5.000000e-01,8.660254e-01,0
4,2023-02-01,ST0001,A0001,2023,2,1,8.660254e-01,5.000000e-01,0
5,2023-03-01,ST0001,A0001,2023,3,1,1.000000e+00,6.123234e-17,1
6,2023-04-01,ST0001,A0001,2023,4,2,8.660254e-01,-5.000000e-01,1
7,2023-05-01,ST0001,A0001,2023,5,2,5.000000e-01,-8.660254e-01,2
8,2023-06-01,ST0001,A0001,2023,6,2,1.224647e-16,-1.000000e+00,2
9,2023-07-01,ST0001,A0001,2023,7,3,-5.000000e-01,-8.660254e-01,2


# 8. Création des lags mensuels

Un lag mensuel = valeur du mois passé.

**Ici les lags ont un sens clair et précis :**
- `Lag_1` = quantité du mois précédent
- `Lag_3` = quantité il y a 3 mois (même trimestre l'année passée)  
- `Lag_12` = quantité du même mois l'année passée ← **très puissant pour la saisonnalité**

C'est exactement la même information que le modèle SARIMA utilise,
ce qui rend la comparaison équitable.

In [9]:
# LAGS MENSUELS
# Calculés par groupe (station × article) — ordre chronologique garanti

group_cols = ['Code Station', 'N° article']

# Lag_1  : mois précédent
df_monthly['Lag_1']  = (
    df_monthly.groupby(group_cols)['quantite_mois'].shift(1)
)

# Lag_3  : il y a 3 mois (même trimestre l'an passé)
df_monthly['Lag_3']  = (
    df_monthly.groupby(group_cols)['quantite_mois'].shift(3)
)

# Lag_12 : même mois l'année passée — capture la saisonnalité annuelle
df_monthly['Lag_12'] = (
    df_monthly.groupby(group_cols)['quantite_mois'].shift(12)
)

# Vérification NaN générés
print("NaN par lag (normaux — début de série) :")
for col in ['Lag_1', 'Lag_3', 'Lag_12']:
    n_nan = df_monthly[col].isna().sum()
    n_tot = len(df_monthly)
    print(f"  {col:8} : {n_nan:,} NaN / {n_tot:,} lignes")


NaN par lag (normaux — début de série) :
  Lag_1    : 187 NaN / 7,631 lignes
  Lag_3    : 558 NaN / 7,631 lignes
  Lag_12   : 2,158 NaN / 7,631 lignes


# 9. Création des rolling windows mensuelles

Les fenêtres glissantes capturent la **tendance récente** sur les N derniers mois.

- `Rolling_3m`  : moyenne des 3 derniers mois (tendance court terme)
- `Rolling_6m`  : moyenne des 6 derniers mois (tendance moyen terme)
- `Rolling_12m` : moyenne des 12 derniers mois (tendance annuelle)
- `Rolling_Std_3m` : volatilité court terme (stations erratiques)
  
Le `shift(1)` avant le rolling est **obligatoire** pour exclure le mois courant.

In [10]:
# ROLLING WINDOWS MENSUELLES
# shift(1) obligatoire → exclure le mois courant du calcul

g = df_monthly.groupby(group_cols)['quantite_mois']

# Moyenne mobile 3 mois (tendance court terme)
df_monthly['Rolling_3m']  = g.transform(
    lambda x: x.shift(1).rolling(3,  min_periods=2).mean()
)

# Moyenne mobile 6 mois (tendance moyen terme)
df_monthly['Rolling_6m']  = g.transform(
    lambda x: x.shift(1).rolling(6,  min_periods=3).mean()
)

# Moyenne mobile 12 mois (tendance annuelle — équivalent SARIMA)
df_monthly['Rolling_12m'] = g.transform(
    lambda x: x.shift(1).rolling(12, min_periods=6).mean()
)

# Écart-type 3 mois (volatilité court terme)
df_monthly['Rolling_Std_3m'] = g.transform(
    lambda x: x.shift(1).rolling(3, min_periods=2).std()
)

# Ratio tendance court / long (accélération ou décélération)
df_monthly['ratio_tendance'] = (
    df_monthly['Rolling_3m'] / (df_monthly['Rolling_12m'] + 1)
).round(4)

print("Rolling windows créées :")
cols_roll = ['Rolling_3m','Rolling_6m','Rolling_12m','Rolling_Std_3m','ratio_tendance']
print(df_monthly[cols_roll].describe().round(1))


Rolling windows créées :
       Rolling_3m  Rolling_6m  Rolling_12m  Rolling_Std_3m  ratio_tendance
count      7257.0      7073.0       6525.0          7257.0          6525.0
mean      36309.8     36775.0      37925.2          9681.4             1.0
std       38142.2     37226.4      36986.1         14178.7             0.3
min        2000.0      2000.0       2416.7             0.0             0.2
25%       14333.3     15000.0      15833.3          2886.8             0.8
50%       24333.3     25000.0      26083.3          6806.9             0.9
75%       46000.0     47333.3      49833.3         11547.0             1.1
max      675000.0    485500.0     363333.3        507029.6             3.1


# 10. Encodage des variables catégorielles

In [11]:
# ENCODAGE STATION ET PRODUIT

from sklearn.preprocessing import LabelEncoder

le_station = LabelEncoder()
le_produit  = LabelEncoder()

df_monthly['StationEncoded'] = le_station.fit_transform(df_monthly['Code Station'])
df_monthly['ProduitEncoded']  = le_produit.fit_transform(df_monthly['N° article'])

# Sauvegarder le mapping pour décoder plus tard
mapping_station = pd.DataFrame({
    'Code Station'   : le_station.classes_,
    'StationEncoded' : range(len(le_station.classes_))
})
mapping_produit = pd.DataFrame({
    'N° article'    : le_produit.classes_,
    'ProduitEncoded': range(len(le_produit.classes_))
})

print("Mapping stations :")
display(mapping_station.head())
print("\nMapping produits :")
display(mapping_produit)


Mapping stations :


,Code Station,StationEncoded
0,ST0001,0
1,ST0002,1
2,ST0003,2
3,ST0004,3
4,ST0005,4



Mapping produits :


,N° article,ProduitEncoded
0,A0001,0
1,A0002,1


# 11. Définition des features et nettoyage final

On définit la liste des features finales et on supprime les NaN
générés par les lags et rolling en début de série.

In [12]:
# FEATURES FINALES ET VARIABLE CIBLE

FEATURES = [
    # Identifiants encodés
    'StationEncoded',
    'ProduitEncoded',

    # Temporelles
    'annee',
    'num_mois',
    'trimestre',
    'mois_sin',
    'mois_cos',
    'saison_benin',

    # Lags mensuels
    'Lag_1',          # mois précédent
    'Lag_3',          # il y a 3 mois
    'Lag_12',         # même mois l'année passée

    # Rolling windows
    'Rolling_3m',     # tendance 3 mois
    'Rolling_6m',     # tendance 6 mois
    'Rolling_12m',    # tendance annuelle
    'Rolling_Std_3m', # volatilité court terme
    'ratio_tendance', # accélération demande

    # Métier
    'nb_commandes',   # nb de jours commandés dans le mois
]

TARGET = 'quantite_mois'

print(f"Nombre de features : {len(FEATURES)}")
print()
print("ÉTAT AVANT NETTOYAGE")
print(f"Dimensions : {df_monthly.shape[0]:,} lignes x {df_monthly.shape[1]} colonnes")
print()
print("NaN par feature :")
print(df_monthly[FEATURES + [TARGET]].isnull().sum().to_string())


Nombre de features : 17

ÉTAT AVANT NETTOYAGE
Dimensions : 7,631 lignes x 24 colonnes

NaN par feature :
StationEncoded       0
ProduitEncoded       0
annee                0
num_mois             0
trimestre            0
mois_sin             0
mois_cos             0
saison_benin         0
Lag_1              187
Lag_3              558
Lag_12            2158
Rolling_3m         374
Rolling_6m         558
Rolling_12m       1106
Rolling_Std_3m     374
ratio_tendance    1106
nb_commandes         0
quantite_mois        0


In [16]:
# SUPPRESSION DES NaN
# On supprime les lignes où Lag_12 est NaN (lag le plus long)
# → garantit que tous les autres lags sont aussi disponibles

print(f"Lignes avant nettoyage : {len(df_monthly):,}")

df_ml = df_monthly.dropna(subset=FEATURES).copy()
df_ml = df_ml.reset_index(drop=True)

print(f"Lignes après nettoyage : {len(df_ml):,}")
print(f"Lignes supprimées      : {len(df_monthly) - len(df_ml):,} (début de série — normal)")
print(f"NaN restants           : {df_ml[FEATURES].isnull().sum().sum()}")
print()
print("Stations conservées :", df_ml['Code Station'].nunique())
print("Articles conservés  :", df_ml['N° article'].nunique())

# Vérifier que toutes les stations survivent
stations_avant = df_monthly['Code Station'].nunique()
stations_apres = df_ml['Code Station'].nunique()
if stations_avant == stations_apres:
    print(f"\n Toutes les {stations_apres} stations conservées")
else:
    perdues = set(df_monthly['Code Station'].unique()) - set(df_ml['Code Station'].unique())
    print(f"\n Stations perdues (historique trop court) : {perdues}")

    # afficharge des collones
pd.DataFrame(df_ml.columns, columns=['Colonnes'])


Lignes avant nettoyage : 7,631
Lignes après nettoyage : 5,473
Lignes supprimées      : 2,158 (début de série — normal)
NaN restants           : 0

Stations conservées : 89
Articles conservés  : 2

 Stations perdues (historique trop court) : {'ST0063', 'ST0071', 'ST0048', 'ST0035', 'ST0034'}


,Colonnes
0,Code Station
1,N° article
2,mois_periode
3,quantite_mois
4,nb_commandes
5,mois
6,annee
7,num_mois
8,trimestre
9,mois_sin


# 12. Découpage train / test temporel

Pour les séries temporelles, le découpage est **toujours temporel** — jamais aléatoire.

Ce découpage est identique à celui utilisé dans ARIMA/SARIMA → comparaison équitable.

In [14]:
# DÉCOUPAGE TEMPOREL TRAIN / TEST
# Identique au découpage ARIMA/SARIMA pour comparaison équitable

# Coupure à 80% dans le temps
date_coupure = df_ml['mois'].quantile(0.80)
# ou fixer manuellement : date_coupure = pd.Timestamp('2023-01-01')

df_train = df_ml[df_ml['mois'] <= date_coupure].copy()
df_test  = df_ml[df_ml['mois'] >  date_coupure].copy()

print(f"Date de coupure : {date_coupure.date()}")
print()
print(f"Train : {len(df_train):,} lignes")
print(f"  Période : {df_train['mois'].min().date()} → {df_train['mois'].max().date()}")
print()
print(f"Test  : {len(df_test):,} lignes")
print(f"  Période : {df_test['mois'].min().date()} → {df_test['mois'].max().date()}")
print()

X_train = df_train[FEATURES]
y_train = df_train[TARGET]
X_test  = df_test[FEATURES]
y_test  = df_test[TARGET]

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")


Date de coupure : 2025-05-01

Train : 4,492 lignes
  Période : 2021-01-01 → 2025-05-01

Test  : 981 lignes
  Période : 2025-06-01 → 2025-12-01

X_train : (4492, 17)
X_test  : (981, 17)


# 13. Vérification finale et export

In [16]:
# VÉRIFICATION FINALE

print("DATASET MENSUEL — PRÊT POUR LA ML")
print(f"Dimensions     : {df_ml.shape[0]:,} lignes x {df_ml.shape[1]} colonnes")
print(f"Période        : {df_ml['mois'].min().date()} → {df_ml['mois'].max().date()}")
print(f"Stations       : {df_ml['Code Station'].nunique()}")
print(f"Articles       : {df_ml['N° article'].nunique()}")
print(f"Features       : {len(FEATURES)}")
print(f"NaN restants   : {df_ml[FEATURES].isnull().sum().sum()}")
print()
print("Statistiques de la variable cible (quantite_mois) :")
print(df_ml[TARGET].describe().round(0))
print()
print("Aperçu final :")
display(df_ml[['mois', 'Code Station', 'N° article', TARGET,
               'Lag_1', 'Lag_12', 'Rolling_3m', 'Rolling_12m',
               'saison_benin', 'nb_commandes']].head(10))


DATASET MENSUEL — PRÊT POUR LA ML
Dimensions     : 5,473 lignes x 24 colonnes
Période        : 2021-01-01 → 2025-12-01
Stations       : 89
Articles       : 2
Features       : 17
NaN restants   : 0

Statistiques de la variable cible (quantite_mois) :
count       5473.0
mean       36393.0
std        40455.0
min         1000.0
25%        15000.0
50%        25000.0
75%        46000.0
max      1078000.0
Name: quantite_mois, dtype: float64

Aperçu final :


,mois,Code Station,N° article,quantite_mois,Lag_1,Lag_12,Rolling_3m,Rolling_12m,saison_benin,nb_commandes
0,2023-10-01,ST0001,A0001,79000,20000.0,20000.0,65000.000000,57166.666667,3,4
1,2023-11-01,ST0001,A0001,70000,79000.0,24000.0,59666.666667,62083.333333,3,4
2,2023-12-01,ST0001,A0001,55000,70000.0,49000.0,56333.333333,65916.666667,0,4
3,2024-01-01,ST0001,A0001,55000,55000.0,56000.0,68000.000000,66416.666667,0,5
4,2024-02-01,ST0001,A0001,105000,55000.0,34000.0,60000.000000,66333.333333,0,7
5,2024-03-01,ST0001,A0001,87000,105000.0,90000.0,71666.666667,72250.000000,1,5
6,2024-04-01,ST0001,A0001,93000,87000.0,48000.0,82333.333333,72000.000000,1,5
7,2024-05-01,ST0001,A0001,143000,93000.0,105000.0,95000.000000,75750.000000,2,9
8,2024-06-01,ST0001,A0001,103000,143000.0,65000.0,107666.666667,78916.666667,2,8
9,2024-07-01,ST0001,A0001,96000,103000.0,95000.0,113000.000000,82083.333333,2,7


In [18]:
# EXPORT

df_ml.to_csv("../data/propre/dataset_ml_mensuel.csv",
             index=False, encoding="utf-8-sig")

print("Dataset exporté : dataset_ml_mensuel.csv")
print()
print("Ce fichier est prêt pour :")
print("  → XGBoost mensuel")
print("  → Random Forest mensuel")
print("  → Comparaison directe avec ARIMA/SARIMA")


Dataset exporté : dataset_ml_mensuel.csv

Ce fichier est prêt pour :
  → XGBoost mensuel
  → Random Forest mensuel
  → Comparaison directe avec ARIMA/SARIMA


## Ce qu'on a fait et pourquoi

**Différence clé avec la version hebdomadaire :**

| | Hebdomadaire | Mensuel |
|---|---|---|
| Granularité | 1 ligne = 1 semaine | 1 ligne = 1 mois |
| Variable cible | `quantite_semaine` | `quantite_mois` |
| Lags | Lag_1, Lag_7 | Lag_1, Lag_3, Lag_12 |
| Rolling | 4sem, 12sem | 3m, 6m, 12m |
| Comparaison ARIMA |  Pas équitable | Même échelle |

**Lag_12 est la feature clé de cette version** — il donne au modèle la valeur
du même mois l'année passée, ce qui est exactement ce que SARIMA(P,D,Q)12 modélise
via sa partie saisonnière. La comparaison entre les modèles est donc directement
équitable.